# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will learn to access metadata, enumerate record sets and fields by `@id`, extract tabular data by `@id`, and apply common exploratory analytics.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/1.0) with the following schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

# Display metadata information (name and description)
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Enumerate available record sets, fields, and their `@id`s from the Croissant schema. Record sets, fields and columns are uniquely identified by their `@id` fields.


In [ ]:
# Get all record sets and explore their `@id`s and field structure

record_sets = dataset.record_sets

print("Available record sets and their fields by @id:\n")
for rs in record_sets:
    print(f"RecordSet: {rs.id}")
    for field in rs.fields:
        print(f"  Field: {field.id}  (data type: {field.data_type})")
    print("")

## 3. Data Extraction
Load records from the main record set(s) into DataFrame(s) for exploration. Use only `@id` values to reference record sets and fields. The example below extracts all rows from each record set.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Each record is a dict with field @id keys
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"[Warning] Record set {record_set_id} returned no data!")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded record set: {record_set_id}, shape: {df.shape}")
    print(f"Field @ids as columns: {df.columns.tolist()}")

# List dataframe keys (record set @ids with data)
print("\nExtracted dataframes for record sets:")
print(list(dataframes.keys()))

# Preview the first dataframe
main_record_set_id = next(iter(dataframes.keys()))
print(f"\nPreview of main record set: {main_record_set_id}")
display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply standard data processing — filter, normalize, and group by key attributes. Use field `@id`s only.

In [ ]:
# Select main dataframe for EDA
df = dataframes[main_record_set_id]

# Let's find numeric fields by their data type from the schema
numeric_fields = []
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if field.data_type in ('Integer', 'Float', 'Number', 'schema:Integer', 'schema:Float', 'schema:Number'):
                numeric_fields.append(field.id)
print("Numeric field @ids:", numeric_fields)

if not numeric_fields:
    raise Exception("No numeric fields detected in the main record set.")

# For illustration, pick the first numeric field
numeric_field_id = numeric_fields[0]

# Filter out records with values above the median for this field
if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows\n")

    # Normalize this field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Top rows for field '{numeric_field_id}' and normalized values:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Numeric field '{numeric_field_id}' is not present or not numeric in the dataframe.")

# Attempt grouping by a categorical field (e.g., first available non-numeric)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
        break

if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','count'])
    print(f"\nMean and count of '{numeric_field_id}' grouped by '{group_field_id}':")
    display(grouped_df.head())
else:
    print("No suitable categorical group field found for grouping.")


## 5. Visualization
Visualize data distributions or relationships between selected fields using their `@id`s. This step demonstrates basic visual EDA with matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If grouping succeeded, plot group means
if ('grouped_df' in locals()) and (group_field_id is not None):
    plt.figure(figsize=(9,4))
    grouped_df['mean'].plot(kind='bar')
    plt.title(f"Group mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook demonstrated how to:

- Load a FAIR-formatted Croissant dataset with `mlcroissant`
- Explore metadata, record set and field structure via `@id`
- Extract tabular record data by `@id` and perform basic data exploration
- Apply filtering, normalization, grouping, and visualization by field `@id`

For further analysis, you can investigate other record sets, explore richer relationships between fields, or build predictive models using the loaded DataFrames.